# Lipschitz of GELU (numeric solution)

In [1]:
import math
import torch

# f(x) = x * Phi(x),  Phi = standard normal CDF
# f'(x) = Phi(x) + x * phi(x),  phi(x) = N(0,1) pdf
def gelu_exact_prime(x):
    # Phi(x) = 0.5 * (1 + erf(x / sqrt(2)))
    Phi = 0.5 * (1.0 + torch.erf(x / math.sqrt(2.0)))
    phi = torch.exp(-0.5 * x * x) / math.sqrt(2.0 * math.pi)
    return Phi + x * phi

# Tanh-approx GELU (Hendrycks & Gimpel) 
# gelu_tanh(x) = 0.5 x [1 + tanh(√(2/π) (x + 0.044715 x^3))]
# derivative derived analytically to avoid autograd memory
def gelu_tanh_prime(x):
    c = math.sqrt(2.0 / math.pi)
    a = 0.044715
    u = c * (x + a * x**3)              # inner argument to tanh
    t = torch.tanh(u)
    du = c * (1.0 + 3.0 * a * x**2)     # derivative of u wrt x
    # d/dx [0.5 x (1 + t)] = 0.5(1 + t) + 0.5 x * (1 - t^2) * du
    return 0.5 * (1.0 + t) + 0.5 * x * (1.0 - t**2) * du

# Generic numeric sup |f'(x)| over [xmin, xmax] 
def lipschitz_numeric(
    deriv_fn,
    xmin = -12.0,
    xmax = 12.0,
    steps = 1_000_001,     
    batch_size = 250_000,
):
    assert xmax > xmin and steps >= 2
    K_max = torch.tensor(0.0, dtype=torch.float64)
    x_at_max = torch.tensor(float("nan"), dtype=torch.float64)

    start = 0
    while start < steps:
        end = min(start + batch_size, steps)
        xs = torch.linspace(
            xmin + (xmax - xmin) * (start / (steps - 1)),
            xmin + (xmax - xmin) * ((end - 1) / (steps - 1)),
            end - start,
            dtype=torch.float64,
        )
        vals = deriv_fn(xs).abs()
        local_max, idx = torch.max(vals, dim=0)
        if local_max > K_max:
            K_max = local_max
            x_at_max = xs[idx]
        start = end

    return K_max.item(), x_at_max.item()

# GELU
K_exact, x_exact = lipschitz_numeric(
        gelu_exact_prime, 
        xmin=-12, xmax=12, 
        steps=1_000_001,
        batch_size=250_000, 
    )

print(f"GELU K = {K_exact:.10f} at x = {x_exact:.6f}")

# Tanh-approx GELU
K_tanh, x_tanh = lipschitz_numeric(
        gelu_tanh_prime, xmin=-10, xmax=10, steps=2_000_001,
        batch_size=250_000,
    )
print(f"Tanh GELU K = {K_tanh:.10f} at x = {x_tanh:.6f}")

GELU K = 1.1289041452 at x = 1.414224
Tanh GELU K = 1.1289930687 at x = 1.418500
